In [2]:
import os
import pandas as pd

In [7]:
lc_file = '/home/alex/Data/Work/Sources/dash_dummy_data/lsst_RRLyr.pkl'

In [8]:
lcs = pd.read_pickle(lc_file)
print(lcs.head())

  band  ccdVisitId   coord_ra  coord_dec             objectId      psfFlux  \
0    y  1032263018  62.462569  -44.11336  1251384969897480052  -515.183603   
1    y  1033987172  62.462569  -44.11336  1251384969897480052  3151.738459   
2    u   675163080  62.462569  -44.11336  1251384969897480052   183.449123   
3    y   443055067  62.462569  -44.11336  1251384969897480052  -704.848327   
4    u   466722002  62.462569  -44.11336  1251384969897480052   382.472233   

    psfFluxErr     psfMag  ccdVisitId2 band2   expMidptMJD  zeroPoint  
0  1697.218490        NaN   1032263018     y  61100.069706  30.602301  
1  1686.955775  22.653625   1033987172     y  61102.068464  30.606100  
2   209.242045  25.741211    675163080     u  60582.247144  30.469101  
3  1624.400086        NaN    443055067     y  60215.203585  30.612801  
4   278.926670  24.943500    466722002     u  60261.078221  30.461201  


In [9]:
lcs.columns

Index(['band', 'ccdVisitId', 'coord_ra', 'coord_dec', 'objectId', 'psfFlux',
       'psfFluxErr', 'psfMag', 'ccdVisitId2', 'band2', 'expMidptMJD',
       'zeroPoint'],
      dtype='object')

In [10]:
lcs['objectId'].nunique()

25

In [11]:
def add_mock_diaSourceId(
    df: pd.DataFrame,
    object_col: str = "objectId",
    time_col: str = "expMidptMJD",
    source_name: str = "LSST",
) -> pd.DataFrame:
    """
    Add mock diaSourceId and source columns.

    diaSourceId format:
        int("1" + object_index(2 digits) + observation_index(4 digits))

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe with multiple objects.
    object_col : str
        Column identifying objects.
    time_col : str
        Time column used to order observations.
    source_name : str
        Value to put into 'source' column.

    Returns
    -------
    df_out : pandas.DataFrame
        Copy of df with added columns:
        - diaSourceId
        - source
    """
    df = df.copy()

    # stable object ordering
    object_ids = sorted(df[object_col].unique())

    dia_ids = []

    for obj_idx, obj_id in enumerate(object_ids, start=1):
        df_obj = df[df[object_col] == obj_id].sort_values(time_col)

        for obs_idx, row_idx in enumerate(df_obj.index, start=1):
            dia_id_str = f"1{obj_idx:02d}{obs_idx:04d}"
            dia_ids.append((row_idx, int(dia_id_str)))

    dia_id_series = pd.Series(
        {idx: dia_id for idx, dia_id in dia_ids},
        name="diaSourceId",
    )

    df["diaSourceId"] = dia_id_series
    df["source"] = source_name

    return df


In [12]:
df2 = add_mock_diaSourceId(lcs)

In [13]:
df2.head()

,band,ccdVisitId,coord_ra,coord_dec,objectId,psfFlux,psfFluxErr,psfMag,ccdVisitId2,band2,expMidptMJD,zeroPoint,diaSourceId,source
0,y,1032263018,62.462569,-44.11336,1251384969897480052,-515.183603,1697.218490,NaN,1032263018,y,61100.069706,30.602301,1010354,LSST
1,y,1033987172,62.462569,-44.11336,1251384969897480052,3151.738459,1686.955775,22.653625,1033987172,y,61102.068464,30.606100,1010356,LSST
2,u,675163080,62.462569,-44.11336,1251384969897480052,183.449123,209.242045,25.741211,675163080,u,60582.247144,30.469101,1010184,LSST
3,y,443055067,62.462569,-44.11336,1251384969897480052,-704.848327,1624.400086,NaN,443055067,y,60215.203585,30.612801,1010110,LSST
4,u,466722002,62.462569,-44.11336,1251384969897480052,382.472233,278.926670,24.943500,466722002,u,60261.078221,30.461201,1010128,LSST


In [14]:

assert df2["diaSourceId"].is_unique
assert df2["source"].unique().tolist() == ["LSST"]

In [15]:

df2[["objectId", "expMidptMJD", "band", "diaSourceId"]].head()


,objectId,expMidptMJD,band,diaSourceId
0,1251384969897480052,61100.069706,y,1010354
1,1251384969897480052,61102.068464,y,1010356
2,1251384969897480052,60582.247144,u,1010184
3,1251384969897480052,60215.203585,y,1010110
4,1251384969897480052,60261.078221,u,1010128


In [19]:
mask = df2['objectId'] != 1251384969897480052
df2[mask]['diaSourceId'].sort_values()

1643     1020001
2065     1020002
1813     1020003
1391     1020004
1983     1020005
          ...   
10776    1250447
11127    1250448
11130    1250449
11157    1250450
11066    1250451
Name: diaSourceId, Length: 10722, dtype: int64

In [20]:
df2.to_pickle('/home/alex/Data/Work/Sources/dash_dummy_data/lsst_RRLyr_with_diaSourceId.pkl')